# Supp Tables 1b and 1c: pathology annotation labels and clinical variables

**Produces**
- **Supp Table 1b** (`1b_pathology_variables`): the 11 pathology annotation labels of the Visium
  spots, after the label standardisation of `pathology_annotation_loader` (typos and variants
  merged; folds, shifted spots, empty tissue, nan and arachnoid removed), in order of first
  appearance.
- **Supp Table 1c** (`1c_clinical_variables`): the 255 variables of the processed clinical
  tables: `center`, the block ID, the 249 columns of the tables, and the four day-unit survival
  columns the notebook derived (`os_days`, `pfs_days`, `os_days_signed`, `pfs_days_signed`).

**Origin**
- 1b: the annotation loading of `fig4e_suppfig4e_colocalization_plots.ipynb` (analysis/cell2loc/
  colocalization_plots.ipynb, Owkin): `pathology_annotation_loader.load_and_prepare` with
  `keep_only_shared_annotations=False`; the table is the list of distinct labels.
- 1c: `analysis/scratch/survival/survival_patient_features.ipynb` (Owkin), `load_clinical()` and the
  commented line `df_clin_processed.columns.to_frame().to_csv('1c_clinical_variables.csv', index=False)`
  (the published file has no header line).

**Compared with the published tables** (EFS `supplementary_tables/`): both identical (same
values, same order).

In [ ]:
import pandas as pd

from gbmcollaboration import AllDataCenters
from gbmcollaboration.definitions import AVG_NUM_DAYS_PER_YEAR
from gbmcollaboration.paper import config, output
from gbmcollaboration.utils import pathology_annotation_loader as patho

In [ ]:
OUT = output.setup("supptables_1bc_pathology_clinical_variables")
PUBLISHED_TABLES = config.EFS / "supplementary_tables"

In [ ]:
def load_clinical():
    """Processed clinical tables of the three centres, as survival_patient_features loaded them."""
    df_clin_processed = (AllDataCenters.load_clinical(exclude_idh1_mut=True)
                         .reset_index())
    df_clin_processed.insert(0, "center", df_clin_processed.pop("center"))
    # add days unit and signed format (-1 for censored)
    df_clin_processed['os_days'] = df_clin_processed['os_years'] * AVG_NUM_DAYS_PER_YEAR
    df_clin_processed['pfs_days'] = df_clin_processed['pfs_years'] * AVG_NUM_DAYS_PER_YEAR
    df_clin_processed['os_days_signed'] = df_clin_processed['os_days'] * (df_clin_processed['os_event'].apply(lambda e: 1 if e == 1 else -1))
    df_clin_processed['pfs_days_signed'] = df_clin_processed['pfs_days'] * (df_clin_processed['pfs_event'].apply(lambda e: 1 if e == 1 else -1))
    return df_clin_processed

## Supp Table 1b: pathology annotation labels

In [ ]:
cfg = patho.DataConfig(
    s3_bucket="pa-bi4jtrdsa151p4d3m-jepjxh9qrwgxphe8hd3r6exicahnoeuw1b-s3alias",
    s3_prefixes=(
        "fd319842-9157-4f04-bd4f-7a36ce6a0ba7/Non_mosaic_upload/Mosaic Annotations/",  # CHUV
        "2ca49430-8b5c-4504-ad96-344019c90e84/",  # UKER
    ),
)
acfg = patho.AnalysisConfig(
    min_spots_per_annotation=0,
    keep_only_shared_annotations=False,
)
df_patho_annot = patho.load_and_prepare(cfg, acfg)

table1b = df_patho_annot['pathology_annotations'].drop_duplicates().to_frame()
output.save_table(OUT, "1b_pathology_variables", table1b, panel="Supp Table 1b", index=False, header=False)
print(df_patho_annot['pathology_annotations'].value_counts().to_string())

## Supp Table 1c: clinical variables

In [ ]:
df_clin_processed = load_clinical()
print(f"{len(df_clin_processed)} rows; {df_clin_processed['center'].value_counts().to_dict()}")
table1c = df_clin_processed.columns.to_frame()
output.save_table(OUT, "1c_clinical_variables", table1c, panel="Supp Table 1c", index=False, header=False)

## Comparison with the published tables

In [ ]:
for name, table in [("1b_pathology_variables", table1b), ("1c_clinical_variables", table1c)]:
    published = pd.read_csv(PUBLISHED_TABLES / f"{name}.csv", header=None)[0].tolist()
    ours = table.iloc[:, 0].astype(str).tolist()
    print(f"{name}: {len(ours)} rows (published {len(published)}); identical: {ours == published}")
    if ours != published:
        print("  only published:", [x for x in published if x not in ours])
        print("  only here:", [x for x in ours if x not in published])

In [ ]:
print(output.versions(["pandas", "boto3"]).to_string(index=False))